# 03 — PROMETHEE II Ranking

**Section:** Evaluation layer (PROMETHEE II, CART baseline, statistical testing, GSEA)
**Depends on:** `01_synthetic_data_foundation.ipynb`, `02_synthetic_rule_pools.ipynb` (`tests/fixtures/synthetic/`, `tests/fixtures/synthetic/rules/`)
**Feeds into:** `04_statistical_testing.ipynb` (Mann–Whitney U on φ), `06_cart_baseline.ipynb` (CART rules join this same unified run once built)

## Recommended reading before this notebook

- **Brans & Vincke, "A preference ranking organisation method: the PROMETHEE method for
  multiple criteria decision-making"** — the source method. §4.6.1 of the proposal is built
  directly on this.
- **Yedjour et al., GPRO_RX** — precedent for combining a genetic rule extractor with
  PROMETHEE-based rule ranking; the multi-criteria framing (fidelity / comprehensibility /
  stability as competing criteria rather than a single scalar score) follows this lineage.
- **CONTRACTS.md §0.4 / §5** — the rule-pool schema this notebook reads, and Arm 5's
  acceptance criterion (PROMETHEE runs as a **single unified execution**, not per-pool).
- **Orientation doc, Part C** — the locked decision that combined-pool PROMETHEE is
  non-negotiable: φ (net outranking flow) values from separate PROMETHEE runs are not
  numerically comparable, so baseline-vs-SAM or GA-vs-NSGA comparisons require everything
  to go through pairwise comparison together.

## What this notebook does

Loads all four synthetic rule pools from notebook 02, computes `stability` per rule (the one
metric notebook 01/02 didn't need), and runs a **single** PROMETHEE II ranking over the
combined 60-rule pool so that fidelity, comprehensibility, and stability trade off against
each other consistently across GA/NSGA-II × baseline/SAM. Produces a full ranked table and a
top-10-per-condition table, both saved for notebook 04's statistical test to consume.


In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from pymcdm.methods import PROMETHEE_II

sys.path.insert(0, "..")  # repo root, so `import src...` resolves

from src.common import metrics
from src.common.config import load_config
from src.common.schema import load_rule_pool

cfg = load_config("../configs/pipeline.yaml")
print(f"config_hash: {cfg.config_hash}")

FIXTURE_DIR = Path("../tests/fixtures/synthetic")
RULES_DIR = FIXTURE_DIR / "rules"
REPORT_DIR = FIXTURE_DIR / "reports"  # see §6 below for why this stays out of artifacts/

feature_map = json.load(open(FIXTURE_DIR / "feature_map.json"))

POOL_NAMES = ["ga_baseline", "ga_sam", "nsga_baseline", "nsga_sam"]
pools = {name: load_rule_pool(RULES_DIR / f"{name}.json", feature_map=feature_map) for name in POOL_NAMES}
for name, pool in pools.items():
    print(f"{name}: {len(pool['rules'])} rules, config_hash={pool['meta']['config_hash']}")
    assert pool["meta"]["config_hash"] == cfg.config_hash, "config_hash mismatch -- pools built from a different pipeline.yaml"


config_hash: a2070aedcc19
ga_baseline: 15 rules, config_hash=a2070aedcc19
ga_sam: 15 rules, config_hash=a2070aedcc19
nsga_baseline: 15 rules, config_hash=a2070aedcc19
nsga_sam: 15 rules, config_hash=a2070aedcc19


## 1. Rebuild the discretised feature matrix

`stability()` needs to slice rows by `cancer_type` and re-derive each subgroup's
(rule_preds, model_preds) pair via `metrics.rule_class_preds()` — which in turn needs the
**discretised** (Low/Intermediate/High) feature matrix, not the raw continuous/binary one,
since that's what `evaluate_rule_mask()` compares rule conditions against.

This is the identical discretisation notebook 02 used (z-score thresholds from
`pipeline.yaml`'s `discretisation` block for pathway activity; binary mutation status mapped
directly to Low/High). Rebuilt here rather than imported, since it's a handful of lines and
keeping this notebook runnable standalone against the fixture files (rather than depending on
notebook 02's in-memory state) matches how 01→02 was already structured.


In [2]:
features_df = pd.read_csv(FIXTURE_DIR / "synthetic_features.csv", index_col="cell_line_id")
labels_df = pd.read_csv(FIXTURE_DIR / "synthetic_labels.csv", index_col="cell_line_id")

def discretise_pathway(series: pd.Series, high_z: float, low_z: float) -> pd.Series:
    z = (series - series.mean()) / series.std()
    return pd.Series(
        np.select([z > high_z, z < low_z], ["High", "Low"], default="Intermediate"),
        index=series.index,
    )

def discretise_mutation(series: pd.Series) -> pd.Series:
    return series.map({0: "Low", 1: "High"})

gene_cols = [c for c in features_df.columns if c.startswith("GENE_")]
pathway_cols = [c for c in features_df.columns if c.startswith("PATHWAY_")]

discretised = features_df[gene_cols + pathway_cols].copy()
for c in gene_cols:
    discretised[c] = discretise_mutation(features_df[c])
for c in pathway_cols:
    discretised[c] = discretise_pathway(features_df[c], cfg.discretisation.high_z, cfg.discretisation.low_z)

discretised = discretised.join(labels_df[["cancer_type", "baseline_pred_class", "sam_pred_class"]])
CLASS_COL = {"baseline": "baseline_pred_class", "sam": "sam_pred_class"}

print(f"discretised: {discretised.shape}, {discretised['cancer_type'].nunique()} cancer subtypes")


discretised: (300, 61), 6 cancer subtypes


## 2. Computing `stability` per rule

**Known limitation, checked by running rather than assumed:** the natural choice would be to
compute stability only on the **test** split, keeping with C8 (train-for-search,
test-for-reporting). Tried first — at this synthetic scale (300 cell lines, 60 in the test
split, ~6 cancer subtypes, rules covering 2–5% of rows) every single rule in all four pools
covers **zero** test-split rows in at least 5 of 6 subtypes, because notebook 02's fabricated
`coverage`/`fidelity_f1` figures were themselves computed against the **full** 300-row
dataset, not train-only (see `02`'s `discretised_df` — no split filter). Restricting stability
to test-only would silently produce `NaN` for essentially every rule.

**Resolution for this notebook:** compute stability on the full synthetic dataset, matching
the basis notebook 02 already used for the `coverage`/`fidelity_f1` fields these rule pools
ship with. This keeps all three PROMETHEE criteria internally consistent (same data basis),
at the cost of not yet exercising the real train/test discipline. **Flag for revisit once real
GDSC-scale data lands** (878 cell lines → a test split with enough rows per subtype for
2%-coverage rules to have non-trivial per-subgroup counts) — this is a synthetic-scale
artefact, not a decision to carry forward unexamined.


In [3]:
def rule_stability(rule: dict, condition: str) -> float:
    """Variance of per-subgroup fidelity across cancer_type, via the shared
    metrics.stability(). Returns NaN if fewer than 2 subgroups have non-zero coverage
    (metrics.stability's own floor) -- see markdown above for why that's not expected
    to trigger at this (full-dataset) basis, but guarded anyway since real data may differ."""
    class_col = CLASS_COL[condition]
    rule_preds_by_subgroup, model_preds_by_subgroup = {}, {}
    for cancer_type, subgroup_df in discretised.groupby("cancer_type"):
        mask = metrics.evaluate_rule_mask(subgroup_df, rule["conditions"])
        if mask.sum() == 0:
            continue
        rule_preds, model_preds, _ = metrics.rule_class_preds(
            subgroup_df, rule["conditions"], rule["predicted_class"], class_col=class_col
        )
        rule_preds_by_subgroup[cancer_type] = rule_preds
        model_preds_by_subgroup[cancer_type] = model_preds
    if len(rule_preds_by_subgroup) < 2:
        return np.nan
    return metrics.stability(rule_preds_by_subgroup, model_preds_by_subgroup)


rows = []
for pool_name, pool in pools.items():
    extractor = pool["meta"]["extractor"]
    condition = pool["meta"]["condition"]
    for r in pool["rules"]:
        rows.append({
            "id": r["id"],
            "extractor": extractor,
            "condition": condition,
            "predicted_class": r["predicted_class"],
            "fidelity_f1": r["fidelity_f1"],
            "length": r["length"],
            "coverage": r["coverage"],
            "stability": rule_stability(r, condition),
        })

combined = pd.DataFrame(rows)
n_nan = combined["stability"].isna().sum()
print(f"{len(combined)} rules total, {n_nan} with unresolved stability (< 2 covered subgroups)")
combined.head()


60 rules total, 0 with unresolved stability (< 2 covered subgroups)


,id,extractor,condition,predicted_class,fidelity_f1,length,coverage,stability
0,ga_baseline_0001,ga,baseline,sensitive,1.0000,2,0.0367,0.000000
1,ga_baseline_0002,ga,baseline,sensitive,1.0000,4,0.0200,0.000000
2,ga_baseline_0003,ga,baseline,sensitive,0.4783,2,0.0400,0.061728
3,ga_baseline_0004,ga,baseline,resistant,0.4706,2,0.0300,0.083333
4,ga_baseline_0005,ga,baseline,resistant,0.4615,2,0.0233,0.083333


## 3. PROMETHEE II — single unified run

**Why single:** C-locked (orientation doc Part C, CONTRACTS.md §5) — φ net flows from separate
PROMETHEE executions are not on a comparable scale, so any baseline-vs-SAM or GA-vs-NSGA-II
comparison downstream must be pairwise-comparisons-within-one-run, not two runs compared
after the fact.

**Criteria (3), all already columns on `combined`:**

| Criterion | Direction | Source |
|---|---|---|
| `fidelity_f1` | maximise (benefit) | stored in rule pool, `metrics.fidelity_f1` |
| `length` (comprehensibility) | minimise (cost) | `metrics.comprehensibility()` — **raw length, lower is better**, see orientation doc §I1 |
| `stability` | minimise (cost) | §2 above — lower variance across subtypes = more stable |

**Preference function — provisional, flagged for team sign-off:** `vshape` (Brans & Vincke's
Type III, linear preference over the observed difference range) with `p` = the criterion's
full observed range and `q = 0` (no indifference threshold — every difference counts, however
small). This is the simplest defensible choice absent a specific reason to prefer a threshold
function; it hasn't been discussed with the team yet.

**Weights — also provisional:** fidelity 0.5, stability 0.3, comprehensibility 0.2. Fidelity
weighted highest since it's the primary claim being tested (rule quality tracking the model);
stability next since Red XAI's fragility framing cares about rules that don't just exploit one
subtype; comprehensibility last but non-zero, since FACC still treats it as a real criterion
(Andrews/Diederich/Tickle 1995), just not the dominant one here.

**Action item (per E5's coordination responsibilities):** once this weight vector is agreed,
publish it into `configs/pipeline.yaml` so GA/NSGA-II can optionally report the same φ-facing
numbers, per the orientation doc's "publish your final PROMETHEE weight vector... back into
`pipeline.yaml`" note.


In [4]:
CRITERIA = ["fidelity_f1", "length", "stability"]
WEIGHTS = np.array([0.5, 0.2, 0.3])  # fidelity, comprehensibility, stability -- see markdown
TYPES = np.array([1, -1, -1])         # 1 = benefit (maximise), -1 = cost (minimise)

matrix = combined[CRITERIA].to_numpy(dtype=float)
assert not np.isnan(matrix).any(), "PROMETHEE matrix has NaNs -- resolve stability gaps before ranking"

p = matrix.max(axis=0) - matrix.min(axis=0)  # vshape linear preference over the full observed range
q = np.zeros(len(CRITERIA))

promethee = PROMETHEE_II("vshape", p=p, q=q)
phi = promethee(matrix, WEIGHTS, TYPES)

combined["phi"] = phi
combined = combined.sort_values("phi", ascending=False).reset_index(drop=True)
combined.head(10)[["id", "extractor", "condition", "fidelity_f1", "length", "stability", "phi"]]


,id,extractor,condition,fidelity_f1,length,stability,phi
0,ga_baseline_0001,ga,baseline,1.0000,2,0.000000,0.609571
1,ga_sam_0001,ga,sam,1.0000,2,0.000000,0.609571
2,nsga_sam_0001,nsga,sam,1.0000,2,0.000000,0.609571
3,nsga_baseline_0001,nsga,baseline,1.0000,2,0.000000,0.609571
4,ga_sam_0002,ga,sam,1.0000,3,0.000000,0.558724
5,ga_baseline_0002,ga,baseline,1.0000,4,0.000000,0.507876
6,ga_sam_0009,ga,sam,0.4333,1,0.001338,0.166681
7,nsga_sam_0014,nsga,sam,0.4333,1,0.001338,0.166681
8,ga_sam_0007,ga,sam,0.4536,1,0.043734,0.115306
9,ga_sam_0010,ga,sam,0.4308,2,0.001812,0.112894


## 4. Top-10 per condition

"Condition" here follows E5's next-steps wording (top-10 **per condition**) taken as the four
`(extractor, condition)` groups — `ga_baseline`, `ga_sam`, `nsga_baseline`, `nsga_sam` — since
that's the granularity the rule pools themselves are organised at, and it's what notebook 04's
baseline-vs-SAM / GA-vs-NSGA-II comparisons need. Flag to the team if "condition" was meant as
just baseline/SAM (i.e. top-10 pooling GA+NSGA-II together) — easy to change, one groupby key.


In [5]:
top10 = (
    combined.sort_values("phi", ascending=False)
    .groupby(["extractor", "condition"])
    .head(10)
    .reset_index(drop=True)
)

print(top10.groupby(["extractor", "condition"]).size())
top10[["id", "extractor", "condition", "predicted_class", "fidelity_f1", "length", "stability", "phi"]].head(12)


extractor  condition
ga         baseline     10
           sam          10
nsga       baseline     10
           sam          10
dtype: int64


,id,extractor,condition,predicted_class,fidelity_f1,length,stability,phi
0,ga_baseline_0001,ga,baseline,sensitive,1.0000,2,0.000000,0.609571
1,ga_sam_0001,ga,sam,resistant,1.0000,2,0.000000,0.609571
2,nsga_sam_0001,nsga,sam,resistant,1.0000,2,0.000000,0.609571
3,nsga_baseline_0001,nsga,baseline,resistant,1.0000,2,0.000000,0.609571
4,ga_sam_0002,ga,sam,sensitive,1.0000,3,0.000000,0.558724
5,ga_baseline_0002,ga,baseline,sensitive,1.0000,4,0.000000,0.507876
6,ga_sam_0009,ga,sam,resistant,0.4333,1,0.001338,0.166681
7,nsga_sam_0014,nsga,sam,resistant,0.4333,1,0.001338,0.166681
8,ga_sam_0007,ga,sam,sensitive,0.4536,1,0.043734,0.115306
9,ga_sam_0010,ga,sam,resistant,0.4308,2,0.001812,0.112894


## 5. Sanity check against the planted ground truth

Same check notebook 02 ran on the raw fabricated pools (§6 there) — repeated here on the
**φ-ranked** top-10 instead of the raw fidelity-sorted pools, to confirm ranking by the full
multi-criteria score doesn't lose the signal that simple fidelity-sorting already found.


In [6]:
ground_truth = json.load(open(FIXTURE_DIR / "synthetic_ground_truth_meta.json"))
true_genes = set(ground_truth["true_signal_genes"].keys())
true_pathways = set(ground_truth["true_signal_pathways"].keys())
true_features = true_genes | true_pathways

rule_lookup = {r["id"]: r for pool in pools.values() for r in pool["rules"]}

for (extractor, condition), group in top10.groupby(["extractor", "condition"]):
    touched = set()
    for rid in group["id"]:
        touched.update(c["feature"] for c in rule_lookup[rid]["conditions"])
    overlap = touched & true_features
    print(f"{extractor}_{condition}: top-10 touches {len(touched)} distinct features, "
          f"{len(overlap)}/{len(true_features)} planted signal features recovered: {sorted(overlap)}")


ga_baseline: top-10 touches 15 distinct features, 4/7 planted signal features recovered: ['GENE_12', 'GENE_17', 'PATHWAY_25', 'PATHWAY_30']
ga_sam: top-10 touches 19 distinct features, 3/7 planted signal features recovered: ['GENE_06', 'PATHWAY_19', 'PATHWAY_30']
nsga_baseline: top-10 touches 13 distinct features, 3/7 planted signal features recovered: ['PATHWAY_19', 'PATHWAY_25', 'PATHWAY_30']
nsga_sam: top-10 touches 15 distinct features, 4/7 planted signal features recovered: ['GENE_12', 'PATHWAY_19', 'PATHWAY_25', 'PATHWAY_30']


## 6. Save outputs

**Placement decision, flagged for confirmation:** these outputs are saved under
`tests/fixtures/synthetic/reports/`, not `artifacts/reports/`. The orientation doc's existing
convention ("fabricated, synthetic, or a stand-in for real output goes under
`tests/fixtures/synthetic/` — never under `artifacts/`") was written about *input* fixtures
(notebook 01's data, notebook 02's fabricated rule pools). This notebook extends the same
logic to *outputs derived from* fabricated inputs: a φ-ranking computed over fabricated rule
pools is itself a stand-in, and putting it under `artifacts/reports/` risks it being mistaken
for a real evaluation result once Melisa's/Sakhile's real pools land. This is a call worth
double-checking with the team rather than assuming — easy to relocate if the convention was
meant more narrowly.


In [7]:
REPORT_DIR.mkdir(parents=True, exist_ok=True)

combined.to_csv(REPORT_DIR / "promethee_ranked_rules.csv", index=False)
top10.to_csv(REPORT_DIR / "promethee_top10_per_condition.csv", index=False)

promethee_config = {
    "criteria": CRITERIA,
    "weights": WEIGHTS.tolist(),
    "types": TYPES.tolist(),
    "preference_function": "vshape",
    "p": p.tolist(),
    "q": q.tolist(),
    "config_hash": cfg.config_hash,
    "status": "provisional -- not yet confirmed with team, see §3 markdown",
}
with open(REPORT_DIR / "promethee_config.json", "w") as f:
    json.dump(promethee_config, f, indent=2)

print(f"wrote {len(combined)}-row ranked table, {len(top10)}-row top-10 table, and promethee_config.json -> {REPORT_DIR}")


wrote 60-row ranked table, 40-row top-10 table, and promethee_config.json -> ../tests/fixtures/synthetic/reports


## Known gaps, to address when real data arrives

- **Stability computed on the full dataset, not test-only** (§2) — a synthetic-scale
  limitation, not a methodological decision. Revisit once real data gives the test split
  enough rows per subtype.
- **PROMETHEE weight vector and preference function are provisional** (§3) — need team
  sign-off, then publishing into `configs/pipeline.yaml`.
- **"Top-10 per condition" granularity assumption** (§4) — confirm whether this means the four
  `(extractor, condition)` groups or the two `condition` values pooling both extractors.
- Once Melisa's real GA branch lands, `ga_baseline`/`ga_sam` here (currently fabricated
  stand-ins) get replaced with real output — no notebook changes needed, since this reads
  through the same `schema.load_rule_pool()` interface either way.

## Next up

- **`04_statistical_testing.ipynb`** — Mann–Whitney U on the φ distributions this notebook
  produced (`promethee_ranked_rules.csv`): baseline vs SAM, and GA vs NSGA-II, as two
  independent-samples comparisons (§4.6.2, Derrac et al.'s nonparametric-testing guidance).
- **`06_cart_baseline.ipynb`** — once built, its rules join this same unified PROMETHEE run
  (evolutionary vs CART) rather than getting a separate one.
